In [1]:
import os
import json
import torch
import logging
import requests
from io import BytesIO
from pathlib import Path
from typing import Dict, List, Any, Optional
from dataclasses import dataclass, field
from PIL import Image
import numpy as np

In [2]:
from transformers import (
    AutoProcessor, 
    AutoModelForImageTextToText,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    HfArgumentParser,
)

In [1]:
from transformers.image_utils import load_image
from torch.utils.data import Dataset
from datasets import load_dataset, Dataset as HFDataset

In [3]:
# Setup logging
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [10]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class ModelArguments:
    model_name_or_path: str = field(
        default="ds4sd/SmolDocling-256M-preview",
        metadata={"help": "Path to pretrained model or model identifier from huggingface.co/models"}
    )
    cache_dir: Optional[str] = field(
        default=None,
        metadata={"help": "Where to store the pretrained models downloaded from huggingface.co"}
    )
    use_flash_attention: bool = field(
        default=True,
        metadata={"help": "Whether to use flash attention for training"}
    )

In [8]:
@dataclass
class DataArguments:
    use_docling_matix: bool = field(
        default=True,
        metadata={"help": "Whether to use DoclingMatix dataset from HuggingFace"}
    )
    docling_matix_subset: Optional[str] = field(
        default=None,
        metadata={"help": "Specific subset of DoclingMatix to use (e.g., 'government_reports', 'scientific_articles')"}
    )
    data_percentage: float = field(
        default=0.0001,  # Changed to 1% of the data
        metadata={"help": "Percentage of the dataset to use (0.01 = 1%)"}
    )
    max_samples: Optional[int] = field(
        default=300,  # Will be calculated from percentage
        metadata={"help": "Maximum number of samples to use for training (calculated from percentage)"}
    )
    data_path: str = field(
        default="./data/train.json",
        metadata={"help": "Path to the training data (fallback if not using DoclingMatix)"}
    )
    validation_data_path: Optional[str] = field(
        default="./data/val.json",
        metadata={"help": "Path to the validation data"}
    )
    image_folder: str = field(
        default="./data/images",
        metadata={"help": "Path to the folder containing images"}
    )
    max_length: int = field(
        default=8192,
        metadata={"help": "Maximum sequence length"}
    )
    preprocessing_num_workers: int = field(
        default=4,
        metadata={"help": "Number of workers for data preprocessing"}
    )
    train_split_ratio: float = field(
        default=0.9,
        metadata={"help": "Ratio for train/validation split when using DoclingMatix"}
    )

In [12]:
from dataclasses import dataclass, field
from transformers import TrainingArguments

@dataclass
class CustomTrainingArguments(TrainingArguments):
    output_dir: str = field(default="./output_version2")
    num_train_epochs: float = field(default=3.0)
    per_device_train_batch_size: int = field(default=1)  # Reduced for memory efficiency
    per_device_eval_batch_size: int = field(default=1)
    gradient_accumulation_steps: int = field(default=16)  # Increased to maintain effective batch size
    learning_rate: float = field(default=1e-5)
    weight_decay: float = field(default=0.01)
    warmup_steps: int = field(default=100)
    logging_steps: int = field(default=10)
    save_steps: int = field(default=500)
    eval_steps: int = field(default=500)
    evaluation_strategy: str = field(default="steps")
    save_total_limit: int = field(default=3)
    load_best_model_at_end: bool = field(default=False)
    metric_for_best_model: str = field(default="eval_loss")
    greater_is_better: bool = field(default=False)
    bf16: bool = field(default=True)
    fp16: bool = field(default=False)
    dataloader_pin_memory: bool = field(default=False)
    remove_unused_columns: bool = field(default=False)
    report_to: str = field(default="none")

In [13]:
class DoclingMatrixDataset(Dataset):
    def __init__(self, processor, max_length=8192, subset=None, max_samples=None, split="train"):
        self.processor = processor
        self.max_length = max_length
        
        logger.info(f"Loading {split} split of DoclingMatix...")
        dataset = load_dataset(
            "HuggingFaceM4/DoclingMatix",
            subset,
            split=split,
            streaming=True
        )

        collected = []
        for i, sample in enumerate(dataset):
            if max_samples and i >= max_samples:
                break
            collected.append(sample)
        self.data = HFDataset.from_list(collected)
        logger.info(f"Loaded {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]

        # Load first image
        try:
            image = sample["images"][0]
            if isinstance(image, str):
                image = Image.open(image).convert("RGB")
            else:
                image = image.convert("RGB")
        except:
            image = Image.new("RGB", (224, 224), "white")

        # Extract assistant text
        if "texts" in sample and sample["texts"]:
            target_text = sample["texts"][0].get("assistant", "")
        else:
            target_text = "Unable to extract text."

        if len(target_text) > 4000:
            target_text = target_text[:4000] + "..."

        user_message = "Convert this document page to markdown format."
        messages = [
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_message}]},
            {"role": "assistant", "content": [{"type": "text", "text": target_text}]}
        ]

        prompt = self.processor.apply_chat_template(messages, add_generation_prompt=False, tokenize=False)

        inputs = self.processor(
            text=prompt,
            images=[image],
            return_tensors="pt",
            max_length=self.max_length,
            truncation=True,
            padding=False
        )

        labels = inputs["input_ids"].clone()
        labels[:, : int(labels.shape[1] * 0.6)] = -100  # simple masking

        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "pixel_values": inputs["pixel_values"].squeeze(),
            "pixel_attention_mask": inputs["pixel_attention_mask"].squeeze(),
            "labels": labels.squeeze(),
        }

In [16]:
from typing import Dict, List

import torch
from transformers import AutoProcessor

class SmolDoclingDataCollator:
    """Custom data collator for SmolDocling"""
    
    def __init__(self, processor: AutoProcessor):
        self.processor = processor
        
    def __call__(self, features: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
        # Separate different types of inputs
        input_ids = [f["input_ids"] for f in features]
        attention_masks = [f["attention_mask"] for f in features]
        pixel_values = [f["pixel_values"] for f in features]
        pixel_attention_masks = [f["pixel_attention_mask"] for f in features]
        labels = [f["labels"] for f in features]
        
        # Pad sequences
        max_length = max(len(ids) for ids in input_ids)
        
        padded_input_ids = []
        padded_attention_masks = []
        padded_labels = []
        
        for ids, mask, lbls in zip(input_ids, attention_masks, labels):
            pad_length = max_length - len(ids)
            
            # Pad input_ids and attention_mask
            padded_ids = torch.cat([ids, torch.full((pad_length,), self.processor.tokenizer.pad_token_id)])
            padded_mask = torch.cat([mask, torch.zeros(pad_length)])
            padded_lbl = torch.cat([lbls, torch.full((pad_length,), -100)])
            
            padded_input_ids.append(padded_ids)
            padded_attention_masks.append(padded_mask)
            padded_labels.append(padded_lbl)
        
        return {
            "input_ids": torch.stack(padded_input_ids),
            "attention_mask": torch.stack(padded_attention_masks),
            "pixel_values": torch.stack(pixel_values),
            "pixel_attention_mask": torch.stack(pixel_attention_masks),
            "labels": torch.stack(padded_labels)
        }

In [23]:
from transformers import HfArgumentParser
import torch

# Parse arguments
parser = HfArgumentParser((ModelArguments, DataArguments, CustomTrainingArguments))
model_args, data_args, training_args = parser.parse_args_into_dataclasses(args=[])

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

INFO:__main__:Using device: cuda


In [ ]:
# Load processor and model
logger.info(f"Loading processor and model from {model_args.model_name_or_path}")
processor = AutoProcessor.from_pretrained(
    model_args.model_name_or_path,
    cache_dir=model_args.cache_dir
)

In [29]:
import importlib.util

import torch
from transformers import AutoModelForImageTextToText

model_kwargs = {
    "torch_dtype": torch.bfloat16,
    "device_map": "auto",
    "cache_dir": model_args.cache_dir,
}

if importlib.util.find_spec("flash_attn") is not None:
    model_kwargs["attn_implementation"] = "flash_attention_2"
else:
    print("flash_attn not installed; loading model without FlashAttention2.")

model = AutoModelForImageTextToText.from_pretrained(
    model_args.model_name_or_path,
    **model_kwargs,
)

flash_attn not installed; loading model without FlashAttention2.


INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


In [27]:
    # Ensure processor has pad token
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

In [28]:
    # Create datasets using DoclingMatix
logger.info("Creating datasets from DoclingMatix...")
train_dataset = DoclingMatrixDataset(
    processor=processor,
    max_length=data_args.max_length,
    subset=data_args.docling_matix_subset,
    #data_percentage=data_args.data_percentage,
    max_samples=300,
    split="train",
    #train_split_ratio=data_args.train_split_ratio
)

INFO:__main__:Creating datasets from DoclingMatix...
INFO:__main__:Loading train split of DoclingMatix...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/1104 [00:00<?, ?it/s]

INFO:__main__:Loaded 300 samples


In [30]:
train_dataset

In [ ]:
train_dataset[0]

In [32]:
eval_dataset = DoclingMatrixDataset(
    processor=processor,
    max_length=data_args.max_length,
    subset=data_args.docling_matix_subset,
    #data_percentage=data_args.data_percentage,
    max_samples=10,
    split="train",
    #train_split_ratio=data_args.train_split_ratio
)

INFO:__main__:Loading train split of DoclingMatix...


Resolving data files:   0%|          | 0/1104 [00:00<?, ?it/s]

INFO:__main__:Loaded 10 samples


In [33]:
data_collator = SmolDoclingDataCollator(processor)

In [37]:
from transformers import Trainer

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [36]:
# Train the model
logger.info("Starting training...")
trainer.train()

# Save the final model
logger.info("Saving final model...")
trainer.save_model()
processor.save_pretrained(training_args.output_dir)

# Save training state
trainer.save_state()

logger.info(f"Training completed. Model saved to {training_args.output_dir}")

INFO:__main__:Starting training...


Step,Training Loss
10,22.855900
20,23.149500
30,22.672500
40,21.692700
50,20.791400


INFO:__main__:Saving final model...
INFO:__main__:Training completed. Model saved to ./output_version2


In [38]:
!pwd

/home/rookie/Documents/Minje/smol_docling_generator


In [39]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this page to docling"}
        ]
    },
]

In [40]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [41]:
image = load_image("labeler_data/uploads/2017-361_page_12.png")

In [42]:
# Prepare inputs
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors="pt")
inputs = inputs.to(DEVICE)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


In [43]:
# Generate outputs
generated_ids = model.generate(**inputs, max_new_tokens=8192)
prompt_length = inputs.input_ids.shape[1]
trimmed_generated_ids = generated_ids[:, prompt_length:]
doctags = processor.batch_decode(
    trimmed_generated_ids,
    skip_special_tokens=False,
)[0].lstrip()

In [44]:
from docling_core.types.doc import DoclingDocument
from docling_core.types.doc.document import DocTagsDocument

In [45]:
# Populate document
doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [image])
print(doctags)

<doctag><text><loc_47><loc_24><loc_471><loc_47>Орлогів длэгэрнгуй таэйанд  орлогів ұҳүээү авсэү  эсұхэ  уаэ авахаа  хүээээү  бүй төэйбөршйн бойт үн  цэнээээ эүхэмкиэ. Орлоготой  холбогодэк  гарсан борэлуулалтын  өртөг болон бусад зардылэт  қинхээн  гарсан  хэмжэээээр нь тусгана. ЭнэхүУ тайданд зардылэт  чигүүэрээн нь ангижд  үр  дүнг тощно.</text>
<section_header_level_1><loc_47><loc_62><loc_210><loc_68>Орлогын дәлэгэрнгуй тайданн  баэнгэт 13х</section_header_level_1>
<unordered_list><list_item><loc_67><loc_73><loc_471><loc_90>(a) Орлогын дәлэгэрнгуй тайданн баэнгэтэдээ  эхуайн тайданн  үөд  хамрагдаэх  орлого, зардын  ажил  гүйдэгээг акүкрэл зарүмааа бүртэнэ;</list_item>
<list_item><loc_67><loc_96><loc_471><loc_113>(b) Орлого ба зардылгэгдэрний  үсэн гари буй  шайтгаанааа  иэ  ангилина. Орлогын зүйд  анги тус  бүртэнэ  бүртэлдын зүйд ангий  эхошуулах зармыг бармитлан бүртэнэ;</list_item>
<list_item><loc_67><loc_119><loc_471><loc_136>(c) Бизнесийн тодорхой  чигэлгэлуэээээээээээээээээээ

In [46]:
# create a docling document
doc = DoclingDocument.load_from_doctags(doctags_doc, document_name="Document")

In [47]:
print(doc.export_to_markdown())

Орлогів длэгэрнгуй таэйанд  орлогів ұҳүээү авсэү  эсұхэ  уаэ авахаа  хүээээү  бүй төэйбөршйн бойт үн  цэнээээ эүхэмкиэ. Орлоготой  холбогодэк  гарсан борэлуулалтын  өртөг болон бусад зардылэт  қинхээн  гарсан  хэмжэээээр нь тусгана. ЭнэхүУ тайданд зардылэт  чигүүэрээн нь ангижд  үр  дүнг тощно.

## Орлогын дәлэгэрнгуй тайданн  баэнгэт 13х
